# DABI2 Projektarbeit: Trinkgeldprognose für einen Online-Lieferdienst

## 1. Einleitung

Dieses Jupyter Notebook dokumentiert die Projektarbeit im Rahmen des Moduls "Datenanalyse und Business Intelligence 2". Ziel ist es, eine integrierte Datenpipeline zu entwickeln, die von der Datenaufnahme und -bereinigung über Feature Engineering und Zeitreihenanalyse bis hin zu Machine Learning-basierten Trinkgeldprognosen reicht. Die Pipeline wird mit Prefect orchestriert und berücksichtigt Konzepte aus den Vorlesungen zu Data Warehousing (Küppers) und Algorithmik (Hofmann).

**Fiktiver Geschäftskontext:**
Wir agieren als internes Data Science Team für einen Online-Lieferdienst für Lebensmittel. Ein zentrales Problem für den Lieferdienst ist die Schwankung des Trinkgeldeinkommens der Lieferboten, was zu Unsicherheiten bei der Personalplanung und potenziell zu Unzufriedenheit führen kann.

## 2. Integrierter Anwendungsfall: Dynamic Tip Prediction & Driver Allocation System

Um die operativen Prozesse zu optimieren und die Zufriedenheit der Lieferboten zu steigern, schlagen wir die Implementierung eines "Dynamic Tip Prediction & Driver Allocation System" vor.

### 2.1. Definition des Geschäftsprozesses

Der Prozess zielt darauf ab, die Trinkgeldwahrscheinlichkeit einer Bestellung in Echtzeit zu prognostizieren und diese Information für eine optimierte Fahrerzuweisung sowie für strategische Betriebsentscheidungen zu nutzen.

1. **Bestelleingang:** Eine neue Bestellung wird vom Kunden aufgegeben und im System registriert.

2. **Echtzeit-Feature-Generierung:** Automatisch werden relevante Features für die neue Bestellung generiert. Dazu gehören historische Kundendaten, Informationen über die bestellten Produkte und zeitliche Merkmale (Uhrzeit, Wochentag etc.).

3. **Trinkgeldprognose:** Ein trainiertes Machine Learning-Modell bewertet die generierten Features und prognostiziert die Wahrscheinlichkeit, dass für diese spezifische Bestellung Trinkgeld gegeben wird.

4. **Optimierte Fahrerzuweisung:** Basierend auf der prognostizierten Trinkgeldwahrscheinlichkeit und anderen Faktoren (z.B. aktuelle Auslastung der Fahrer, Lieferdistanz, Wartezeiten) schlägt das System dem Operations Manager eine optimale Zuweisung der Bestellung zu einem verfügbaren Fahrer vor. Dies kann z.B. eine faire Verteilung von "attraktiven" Bestellungen oder die Priorisierung von Fahrern mit geringer Auslastung beinhalten.

5. **Feedback-Schleife & Modell-Update:** Nach Abschluss der Lieferung wird das tatsächliche Trinkgeld erfasst. Diese Information dient als Feedback für die kontinuierliche Verbesserung des Prognosemodells, das regelmäßig neu trainiert wird.

6. **Performance Monitoring & Reporting:** Ein Dashboard (oder wöchentliche Reports) liefert Einblicke in die durchschnittliche Trinkgeldrate, die Modellperformance und mögliche Optimierungspotenziale (z.B. zu welchen Zeiten oder bei welchen Produkten ist die Trinkgeldrate am höchsten).

### 2.2. Festlegung von Stakeholdern und Datenkonsumenten

* **Lieferboten (Fahrer):** Primäre Stakeholder, die von einer stabilen Einkommenssituation profitieren. Sie sind indirekte Konsumenten der optimierten Zuweisungen.

* **Operations Manager:** Verantwortlich für die Einsatzplanung der Lieferboten und die effiziente Abwicklung von Bestellungen. Sie konsumieren die Trinkgeldprognosen für die Zuweisung und aggregierte Reports für die Schichtplanung.

* **Marketing-Abteilung:** Interessiert an Mustern in der Trinkgeldgabe, um z.B. Produktempfehlungen oder Aktionen zu entwickeln, die die Trinkgeldmotivation erhöhen könnten. Sie sind Konsumenten von spezialisierten Analysen und Reports.

* **Controlling/Finanzabteilung:** Benötigt genaue Trinkgeldprognosen für die Umsatzplanung und Analyse der Fahrereinkommen.

* **Kundenbetreuung:** Könnte auf aggregierte Daten zugreifen, um Kundenanfragen bezüglich Trinkgeldern besser zu beantworten oder allgemeine Trends zu verstehen.

* **Data Science / BI-Team (Wir):** Verantwortlich für die Entwicklung, Wartung und Verbesserung der gesamten Datenpipeline und der Prognosemodelle.

### 2.3. Spezifikation der analytischen Anforderungen

Die Kernaufgaben lassen sich in folgende analytische Anforderungen unterteilen, die die Vorgaben aus Teil 1 und unsere Erweiterungen aus Teil 2 integrieren:

* **A1: Datenaufnahme und -bereinigung:** Effizientes Laden und initiale Bereinigung der Quelldaten (`orders`, `tips_public`, `order_products_denormalized`). Standardisierung von Datentypen und Handhabung von unsinnigen Spalten.

* **A2: Data Warehousing (Küppers):** Transformation der operationellen Daten in ein analytisches Sternschema (Dimensionen: Zeit, User, Produkt; Faktentabellen: Bestellungen, Bestellpositionen) zur Unterstützung von Reporting und Ad-hoc-Analysen.

* **A3: Zeitreihenanalyse der Trinkgeldgabe (Teil 1, erweitert):**

  * Untersuchung der univariaten Zeitreihe der Trinkgeldgabe auf Autokorrelationen, partielle Autokorrelationen, Periodizitäten (stündlich, täglich, saisonal) und Trends.

  * Entwicklung eines SARIMAX-Modells zur Vorhersage der zukünftigen durchschnittlichen Trinkgeldwahrscheinlichkeit pro Zeiteinheit (z.B. Stunde), unter Berücksichtigung von identifizierten Periodizitäten und Trends.

* **A4: Feature Engineering (Teil 2):**

  * Identifikation und Generierung weiterer relevanter Merkmale zur Verbesserung der Trinkgeldprognose. Dazu gehören:

    * **Bestellbezogene Features:** Anzahl Produkte, Anzahl einzigartiger Abteilungen/Gänge, Alkohol im Warenkorb, aggregierte Trinkgeldraten pro Abteilung/Gang.

    * **Nutzerbezogene Features:** Anzahl vergangener Bestellungen, durchschnittliche Bestellintervalle, bevorzugte Bestellzeiten (Stunde, Wochentag), aggregierte Trinkgeldraten des Nutzers, verzögerte Trinkgeldinformationen (`tip_lag_1`, `tip_lag_2`, `tip_lag_3`).

    * **Zeitbasierte Features:** Stunde, Wochentag, Wochenende, Jahreszeit (als Sinus/Cosinus-Transformationen für Periodizität).

    * **Prognose-Features:** Die vorhergesagte stündliche Trinkgeldwahrscheinlichkeit aus dem SARIMAX-Modell (`forecasted_hourly_tip_share`).

* **A5: Machine Learning Modellierung (Hofmann):**

  * Training eines Klassifikationsmodells (z.B. RandomForestClassifier) auf dem erweiterten Feature-Set zur Vorhersage der binären Trinkgeldgabe (`yes`/`no`).

  * Evaluation der Modellperformance mittels relevanter Metriken (Accuracy, ROC-AUC, Classification Report).

* **A6: Prognoseerstellung:** Anwendung des trainierten Modells auf die unbekannten Bestellungen, um die finalen Trinkgeldprognosen zu generieren.

* **A7: Reporting und Visualisierung:** Erstellung von Plots und aggregierten Übersichten zur Veranschaulichung von Analysen und Modellergebnissen.

### 2.4. Datenflussmodellierung

Unser Datenfluss ist als eine orchestrierte Pipeline konzipiert, die die Daten von den Quelldateien bis zu den finalen Prognosen und analytischen Erkenntnissen transformiert und bewegt. Prefect dient hierbei als zentrales Orchestrierungswerkzeug, das die Abhängigkeiten zwischen den einzelnen Schritten verwaltet.

```mermaid
graph TD
    A[Raw Data: orders.parquet, tips_public.csv, order_products_denormalized.csv] --> B(ETL: Load & Clean Data);
    B --> C{Cleaned Data};
    C --> D(DWH: Create Tables);
    D --> E(DWH: Load Data);
    E --> F[Data Warehouse (SQLite)];
    C --> G(Time Series Analysis);
    C --> H(Forecasting: SARIMAX Model);
    H --> I[Forecasted Tip Share];
    C --> J(Feature Engineering);
    I --> J;
    J --> K[Feature-Rich DataFrame];
    K --> L(Model Training & Evaluation);
    L --> M[Trained ML Model];
    K --> N(Make Predictions);
    M --> N;
    N --> O[Final Tip Predictions (CSV)];
    F --> P(DWH: Example Query / Reporting);
    G --> Q[Analysis Plots];
```


**Erläuterung des Datenflusses:**

1. **Raw Data:** Die Ausgangsdaten liegen in verschiedenen Formaten vor.

2. **ETL (Load & Clean Data):** Zuerst werden die Rohdaten geladen und grundlegend bereinigt (Datentypkonvertierung, unnötige Spalten entfernen). Dieser Schritt ist in `src/etl.py` implementiert.

3. **Data Warehousing (Küppers-Aspekt):** Die bereinigten Daten werden verwendet, um ein analytisches Data Warehouse zu befüllen.

    * `Create DWH Tables`: Erstellt die Dimensionen- und Faktentabellen in einer SQLite-Datenbank.

    * `Load Data to DWH`: Lädt die bereinigten Daten in diese DWH-Struktur. Dies ermöglicht später OLAP-ähnliche Abfragen für Reporting-Zwecke. (`src/dwh.py`)

4. **Time Series Analysis:** Parallel zur DWH-Befüllung werden die bereinigten Daten für die Zeitreihenanalyse verwendet, um Periodizitäten und Trends zu identifizieren. (`src/analysis.py`)

5. **Forecasting (Hofmann-Aspekt):** Aufbauend auf der Zeitreihenanalyse wird ein SARIMAX-Modell trainiert, um zukünftige Trinkgeldwahrscheinlichkeiten vorherzusagen. Dieser Output (`Forecasted Tip Share`) wird als Feature für das ML-Modell verwendet. (`src/forecasting.py`)

6. **Feature Engineering:** Hier werden die bereinigten Daten und die Zeitreihenprognosen kombiniert, um ein umfassendes Set an Features zu generieren. Dies beinhaltet auch die Implementierung von `tip_lag_X` und `forecasted_hourly_tip_share` als neue Merkmale. (`src/features.py`)

7. **Model Training & Evaluation (Hofmann-Aspekt):** Das umfangreiche Feature-Set wird genutzt, um das RandomForestClassifier-Modell zu trainieren und zu evaluieren. (`src/modeling.py`)

8. **Make Predictions:** Das trainierte Modell wird verwendet, um die finalen Trinkgeldprognosen für die unbekannten Bestellungen zu erstellen. (`src/modeling.py`)

9. **DWH Example Query / Reporting:** Das befüllte Data Warehouse kann für analytische Abfragen und die Erstellung von Reports genutzt werden, die Einblicke für die Stakeholder liefern.

10. **Analysis Plots & Final Tip Predictions:** Die finalen Outputs des Prozesses.

**Orchestrierung mit Prefect:**
Der gesamte Datenfluss wird durch einen Prefect-Flow (`flow.py`) orchestriert. Jeder logische Schritt ist als Prefect Task definiert, was die Modularität, Überwachung und Skalierbarkeit der Pipeline verbessert. Prefect verwaltet die Abhängigkeiten zwischen den Tasks (z.B. Feature Engineering wartet auf das Ergebnis des Forecasts) und ermöglicht eine robuste Ausführung.

## 3. Technische Umsetzung und Methodik

Die Implementierung erfolgt in Python und ist in mehrere Module aufgeteilt, die jeweils spezifische Aufgaben innerhalb der Datenpipeline übernehmen.

### 3.1. Projektstruktur

```
.
├── data/
│   ├── orders.parquet
│   ├── tips_public.csv
│   ├── order_products_denormalized.csv
│   └── dwh.db             # NEU: SQLite Data Warehouse
├── output/
│   ├── Time Series with Forecast.png
│   ├── Hourly Tip Share with Forecast.png
│   ├── Tipping Periodicity Analysis.png
│   ├── Weekly Tip Percentage Trend.png
│   ├── trained_random_forest_model.pkl
│   └── predictions.csv    # NEU: Modellvorhersagen
├── src/
│   ├── __init__.py
│   ├── analysis.py        # Analysen und Visualisierungen
│   ├── etl.py             # Datenextraktion, -transformation, -ladung
│   ├── features.py        # Feature Engineering
│   ├── forecasting.py     # Zeitreihenprognose
│   ├── modeling.py        # Modelltraining und Prognoseerstellung
│   └── dwh.py             # NEU: Data Warehouse Erstellung und Befüllung
├── flow.py                # Prefect Haupt-Workflow
└── notebook.py            # Dieses Notebook
```

### 3.2. ETL-Prozess (`src/etl.py`)

Dieses Modul ist verantwortlich für das Laden der Rohdaten und deren initiale Bereinigung sowie Typkonvertierung.

In [ ]:
# src/etl.py
import pandas as pd
from typing import Dict

def load_data(data_paths: Dict[str, str]) -> Dict[str, pd.DataFrame]:
    """
    Loads all necessary datasets from specified paths.
    
    Args:
        data_paths (Dict[str, str]): A dictionary mapping dataset names to their file paths.
        
    Returns:
        Dict[str, pd.DataFrame]: A dictionary of loaded DataFrames.
    """
    orders = pd.read_parquet(data_paths['orders'])
    tips_public = pd.read_csv(data_paths['tips_public'])
    order_products = pd.read_csv(data_paths['order_products'])
    
    print("Data loaded successfully.")
    return {
        "orders": orders,
        "tips_public": tips_public,
        "order_products": order_products
    }

def clean_and_prepare_data(dataframes: Dict[str, pd.DataFrame]) -> Dict[str, pd.DataFrame]:
    """
    Performs initial cleaning, type conversion, and preprocessing on the dataframes.
    
    Args:
        dataframes (Dict[str, pd.DataFrame]): Dictionary of raw dataframes.
        
    Returns:
        Dict[str, pd.DataFrame]: Dictionary of cleaned dataframes.
    """
    orders = dataframes['orders'].copy()
    tips_public = dataframes['tips_public'].copy()
    order_products = dataframes['order_products'].copy()

    # Clean up column names and drop unnecessary ones
    if "Unnamed: 0" in tips_public.columns:
        tips_public = tips_public.drop(columns=["Unnamed: 0"])
    if "Unnamed: 0" in order_products.columns:
        order_products = order_products.drop(columns=["Unnamed: 0"])
        
    # Standardize data types
    orders['order_id'] = orders['order_id'].astype('int64')
    orders['user_id'] = orders['user_id'].astype('int64')
    orders['order_date'] = pd.to_datetime(orders['order_date'])
    tips_public['order_id'] = tips_public['order_id'].astype('int64')
    order_products['order_id'] = order_products['order_id'].astype('int64')
    order_products['product_id'] = order_products['product_id'].astype('int64')

    # Optimize memory usage with categorical types
    order_products['department_name'] = order_products['department_name'].astype('category')
    order_products['aisle_name'] = order_products['aisle_name'].astype('category')
    
    print("Data cleaned and prepared successfully.")
    return {
        "orders": orders,
        "tips_public": tips_public,
        "order_products": order_products
    }


### 3.3. Data Warehousing / OLAP (`src/dwh.py` - Küppers-Teil)

Dieses neue Modul ist für die Erstellung und Befüllung eines einfachen Data Warehouses in SQLite zuständig. Es implementiert ein Sternschema mit `dim_time`, `dim_user`, `dim_product` Dimensionen und `fact_orders`, `fact_order_products` Faktentabellen.

In [ ]:
# src/dwh.py
import pandas as pd
import sqlite3
import os

def create_dwh_tables(db_path='data/dwh.db'):
    """
    Erstellt die Tabellen für ein einfaches Data Warehouse (Sternschema) in SQLite.
    
    Args:
        db_path (str): Pfad zur SQLite-Datenbankdatei.
    """
    os.makedirs(os.path.dirname(db_path), exist_ok=True) # Stelle sicher, dass das Verzeichnis existiert
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Dimensionstabellen erstellen
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS dim_time (
            time_id INTEGER PRIMARY KEY,
            order_date TEXT NOT NULL,
            hour INTEGER NOT NULL,
            day_of_week TEXT NOT NULL,
            month INTEGER NOT NULL,
            year INTEGER NOT NULL,
            is_weekend INTEGER NOT NULL
        );
    ''')

    cursor.execute('''
        CREATE TABLE IF NOT EXISTS dim_user (
            user_id INTEGER PRIMARY KEY
        );
    ''')

    cursor.execute('''
        CREATE TABLE IF NOT EXISTS dim_product (
            product_id INTEGER PRIMARY KEY,
            product_name TEXT NOT NULL,
            aisle_id INTEGER,
            aisle_name TEXT,
            department_id INTEGER,
            department_name TEXT
        );
    ''')

    # Faktentabelle
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS fact_orders (
            order_id INTEGER PRIMARY KEY,
            user_id INTEGER,
            time_id INTEGER,
            tip_given INTEGER, -- 1 if tipped, 0 if not (-1 for unknown)
            order_total_products INTEGER,
            FOREIGN KEY (user_id) REFERENCES dim_user(user_id),
            FOREIGN KEY (time_id) REFERENCES dim_time(time_id)
        );
    ''')
    
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS fact_order_products (
            order_product_id INTEGER PRIMARY KEY AUTOINCREMENT,
            order_id INTEGER,
            product_id INTEGER,
            add_to_cart_order INTEGER,
            FOREIGN KEY (order_id) REFERENCES fact_orders(order_id),
            FOREIGN KEY (product_id) REFERENCES dim_product(product_id)
        );
    ''')

    conn.commit()
    conn.close()
    print(f"Data Warehouse-Tabellen in '{db_path}' erfolgreich erstellt oder aktualisiert.")


def load_data_to_dwh(cleaned_data: dict, db_path='data/dwh.db'):
    """
    Lädt die bereinigten Daten in das Data Warehouse.
    
    Args:
        cleaned_data (dict): Dictionary mit den bereinigten DataFrames ('orders', 'tips_public', 'order_products').
        db_path (str): Pfad zur SQLite-Datenbankdatei.
    """
    conn = sqlite3.connect(db_path)
    
    orders_df = cleaned_data['orders'].copy()
    tips_public_df = cleaned_data['tips_public'].copy()
    order_products_df = cleaned_data['order_products'].copy()

    # --- Dim_Time befüllen ---
    print("Befülle dim_time...")
    time_data = orders_df[['order_date']].drop_duplicates().copy()
    time_data['hour'] = time_data['order_date'].dt.hour
    time_data['day_of_week'] = time_data['order_date'].dt.day_name()
    time_data['month'] = time_data['order_date'].dt.month
    time_data['year'] = time_data['order_date'].dt.year
    time_data['is_weekend'] = time_data['order_date'].dt.weekday.isin([5, 6]).astype(int)
    
    # Erstelle eine time_id (einfacher Hash der Zeit, für Demo ausreichend)
    time_data['time_id'] = (time_data['order_date'].astype(int) / 10**9).astype(int) 
    
    time_data_to_load = time_data[['time_id', 'order_date', 'hour', 'day_of_week', 'month', 'year', 'is_weekend']]
    time_data_to_load.to_sql('dim_time', conn, if_exists='append', index=False)
    print("dim_time befüllt.")

    # --- Dim_User befüllen ---
    print("Befülle dim_user...")
    user_data = orders_df[['user_id']].drop_duplicates()
    user_data.to_sql('dim_user', conn, if_exists='append', index=False)
    print("dim_user befüllt.")

    # --- Dim_Product befüllen ---
    print("Befülle dim_product...")
    product_data = order_products_df[['product_id', 'product_name', 'aisle_id', 'aisle_name', 'department_id', 'department_name']].drop_duplicates()
    product_data.to_sql('dim_product', conn, if_exists='append', index=False)
    print("dim_product befüllt.")

    # --- Fact_Orders befüllen ---
    print("Befülle fact_orders...")
    fact_orders_df = orders_df.merge(tips_public_df[['order_id', 'tip']], on='order_id', how='left')
    fact_orders_df['tip_given'] = fact_orders_df['tip'].map({'yes': 1, 'no': 0}).fillna(-1).astype(int) # -1 für unbekannte Tipps
    
    fact_orders_df = fact_orders_df.merge(time_data[['order_date', 'time_id']], on='order_date', how='left')

    order_product_counts = order_products_df.groupby('order_id').size().reset_index(name='order_total_products')
    fact_orders_df = fact_orders_df.merge(order_product_counts, on='order_id', how='left')

    fact_orders_to_load = fact_orders_df[['order_id', 'user_id', 'time_id', 'tip_given', 'order_total_products']]
    fact_orders_to_load.to_sql('fact_orders', conn, if_exists='append', index=False)
    print("fact_orders befüllt.")

    # --- Fact_Order_Products befüllen ---
    print("Befülle fact_order_products...")
    fact_order_products_to_load = order_products_df[['order_id', 'product_id', 'add_to_cart_order']]
    fact_order_products_to_load.to_sql('fact_order_products', conn, if_exists='append', index=False)
    print("fact_order_products befüllt.")

    conn.close()
    print(f"Daten erfolgreich in das Data Warehouse '{db_path}' geladen.")


def query_dwh_example(db_path='data/dwh.db'):
    """
    Führt eine beispielhafte Abfrage auf dem Data Warehouse aus.
    Z.B. durchschnittliche Trinkgeldrate pro Wochentag.
    """
    conn = sqlite3.connect(db_path)
    query = """
    SELECT
        dt.day_of_week,
        AVG(CASE WHEN fo.tip_given = 1 THEN 1.0 ELSE 0.0 END) * 100 AS tip_percentage
    FROM
        fact_orders fo
    JOIN
        dim_time dt ON fo.time_id = dt.time_id
    WHERE
        fo.tip_given != -1 -- Nur bekannte Trinkgeldinformationen berücksichtigen
    GROUP BY
        dt.day_of_week
    ORDER BY
        tip_percentage DESC;
    """
    result = pd.read_sql_query(query, conn)
    conn.close()
    print("\nBeispielhafte DWH-Abfrage (Durchschnittliche Trinkgeldrate pro Wochentag):")
    print(result.to_markdown(index=False)) # Für bessere Darstellung im Notebook
    return result


### 3.4. Zeitreihenanalyse und Prognose (`src/forecasting.py` - Hofmann-Teil)

Dieses Modul behandelt die univariate Zeitreihe der Trinkgeldgabe. Es identifiziert Periodizitäten und Trends und nutzt ein SARIMAX-Modell zur Prognose der zukünftigen Trinkgeldwahrscheinlichkeit.

In [ ]:
# src/forecasting.py
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg') 
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX

def calculate_hourly_tip_shares(orders_df, tips_df):
    orders_df['order_date'] = pd.to_datetime(orders_df['order_date'])
    df = pd.merge(orders_df, tips_df, on='order_id', how='left')
    df['tip'] = df['tip'].map({'yes': 1, 'no': 0}).fillna(0) # Konvertiere tip zu 0/1
    df = df.set_index('order_date')

    hourly_orders = df['tip'].resample('h').count()
    hourly_tip_share = df['tip'].resample('h').mean()

    return pd.DataFrame({
        'tip_share': hourly_tip_share,
        'num_orders': hourly_orders
    })

def trim_stable_time_period(df, min_orders=20):
    valid = df['num_orders'] >= min_orders
    if not valid.any():
        raise ValueError("No time periods with enough orders found.")
    start = df.index[valid.argmax()]
    end = df.index[::-1][valid[::-1].argmax()]
    return df.loc[start:end]

def remove_trend_and_seasonality(df):
    df = df.copy()
    # Einfache Differenzierung zur Entfernung von Trend und Saison (z.B. 24 Stunden für tägliche Saisonalität)
    df['d_tip_share'] = df['tip_share'].diff().diff(24).dropna() 
    return df

def sarimax_forecast(df, steps=24):
    series = df['d_tip_share'].ffill() # Nutze die differenzierte Reihe für SARIMAX

    # Modellparameter können optimiert werden; hier ein Beispiel
    model = SARIMAX(series, order=(1, 1, 1), seasonal_order=(1, 1, 1, 24),
                    enforce_stationarity=False, enforce_invertibility=False)
    result = model.fit(disp=False)
    forecast_diff = result.forecast(steps=steps)

    # Re-Integration der Differenzen
    # Hier ist besondere Vorsicht geboten, da wir zweimal differenziert haben.
    # Für eine einfache Re-Integration fügen wir die letzten bekannten Werte der Originalreihe hinzu.
    # Dies ist eine Vereinfachung; eine exakte Re-Integration erfordert mehr Logik.
    last_known_diff = df['tip_share'].diff().iloc[-1]
    last_known_original = df['tip_share'].iloc[-1]
    
    forecast_original = [last_known_original + last_known_diff + forecast_diff.iloc[0]]
    for i in range(1, steps):
        forecast_original.append(forecast_original[-1] + forecast_diff.iloc[i])
    
    forecast_index = pd.date_range(start=df.index[-1] + pd.Timedelta(hours=1), periods=steps, freq='h')
    forecast_df = pd.DataFrame({
        'tip_share': forecast_original,
        'num_orders': np.nan, # Keine Bestellungen für Prognosezeitraum bekannt
        'd_tip_share': np.nan,
        'is_forecast': True
    }, index=forecast_index)

    df['is_forecast'] = False
    combined = pd.concat([df[['tip_share', 'num_orders', 'd_tip_share', 'is_forecast']], forecast_df])
    return combined

def plot_time_series(before_df, after_df):
    plt.figure(figsize=(12, 6))
    plt.subplot(2, 1, 1)
    plt.plot(before_df['tip_share'], label='Original')
    plt.title("Original Time Series (Trimmed)")
    plt.legend()

    plt.subplot(2, 1, 2)
    plt.plot(after_df['tip_share'], label='With Forecast', color='red')
    plt.title("Time Series with Forecast")
    plt.legend()

    plt.tight_layout()
    plt.savefig('output/Time Series with Forecast.png')
    plt.close() # Close figure to free memory and prevent RuntimeError
    print("Time Series with Forecast saved successfully.")

def process_tip_time_series(orders_df, tips_df, min_orders=20, forecast_hours=24):
    hourly_df = calculate_hourly_tip_shares(orders_df, tips_df)
    trimmed_df = trim_stable_time_period(hourly_df, min_orders)
    detrended_df = remove_trend_and_seasonality(trimmed_df)
    forecast_df = sarimax_forecast(detrended_df, steps=forecast_hours)
    plot_time_series(detrended_df, forecast_df)
    return forecast_df


### 3.5. Feature Engineering (`src/features.py`)

Dieses Modul ist verantwortlich für die Erstellung der umfangreichen Feature-Sets, die für das Machine Learning Modell benötigt werden. Es kombiniert Informationen aus allen Quelldaten und fügt neue, abgeleitete Features hinzu, einschließlich der verzögerten Trinkgeldinformationen (`tip_lag_X`) und der Zeitreihenprognosen (`forecasted_hourly_tip_share`).

In [ ]:
# src/features.py (Auszüge, da das vollständige Modul sehr lang ist)
import numpy as np
import pandas as pd
from IPython.display import display

# Helper functions (omitted for brevity, assume they are present as in your original file)
# ... merge_with_orders, validate_dataframe, add_feature_user_tip_lags ...

# Add feature for alcohol in order
def add_feature_order_alcohol(order_products_df: pd.DataFrame) -> pd.DataFrame:
    """Checks if an order contains alcohol based on department names."""
    alcohol_departments = ['alcohol'] # or other relevant department names
    order_products_df['is_alcohol'] = order_products_df['department_name'].isin(alcohol_departments).astype(int)
    order_has_alcohol = order_products_df.groupby('order_id')['is_alcohol'].max().reset_index()
    order_has_alcohol.rename(columns={'is_alcohol': 'order_has_alcohol'}, inplace=True)
    return order_has_alcohol

# Add features for order product counts and unique departments/aisles
def add_feature_order_product_stats(order_products_df: pd.DataFrame) -> pd.DataFrame:
    """Aggregates product and unique department/aisle counts per order."""
    order_stats = order_products_df.groupby('order_id').agg(
        order_product_count=('product_id', 'count'),
        order_unique_dept_count=('department_id', 'nunique'),
        order_unique_aisle_count=('aisle_id', 'nunique')
    ).reset_index()
    return order_stats

# Add average tip rate per department/aisle
def add_feature_item_tip_rates(order_products_df, tips_public_df, default_tip_rate=0.05):
    """Calculates average tip rate per department and aisle."""
    df_merged = pd.merge(order_products_df, tips_public_df, on='order_id', how='left')
    df_merged['tip_numeric'] = df_merged['tip'].map({'yes': 1, 'no': 0})

    dept_tip_rate = df_merged.groupby('department_id')['tip_numeric'].mean().fillna(default_tip_rate).reset_index(name='order_dept_tip_rate')
    aisle_tip_rate = df_merged.groupby('aisle_id')['tip_numeric'].mean().fillna(default_tip_rate).reset_index(name='order_aisle_tip_rate')

    order_product_info = df_merged[['order_id', 'department_id', 'aisle_id']].drop_duplicates()
    order_product_info = order_product_info.merge(dept_tip_rate, on='department_id', how='left')
    order_product_info = order_product_info.merge(aisle_tip_rate, on='aisle_id', how='left')
    
    # Take the mean rate across all items in an order (simplification)
    order_avg_tip_rates = order_product_info.groupby('order_id').agg(
        order_dept_tip_rate=('order_dept_tip_rate', 'mean'),
        order_aisle_tip_rate=('order_aisle_tip_rate', 'mean')
    ).reset_index()
    return order_avg_tip_rates

# Add time-based features
def add_feature_time_of_order(orders_df: pd.DataFrame) -> pd.DataFrame:
    """Extracts time-based features from order_date."""
    df = orders_df[['order_id', 'order_date']].copy()
    df['order_placed_hour'] = df['order_date'].dt.hour
    df['order_placed_dow'] = df['order_date'].dt.dayofweek # Monday=0, Sunday=6
    df['order_is_weekend'] = df['order_date'].dt.weekday.isin([5, 6]).astype(int)

    # Sin/Cos transformation for cyclical features (hour, dayofweek, month/season)
    df['order_placed_hour_sin'] = np.sin(2 * np.pi * df['order_placed_hour'] / 24)
    df['order_placed_hour_cos'] = np.cos(2 * np.pi * df['order_placed_hour'] / 24)
    df['order_placed_dow_sin'] = np.sin(2 * np.pi * df['order_placed_dow'] / 7)
    df['order_placed_dow_cos'] = np.cos(2 * np.pi * df['order_placed_dow'] / 7)
    
    # Assuming 4 seasons for simplicity
    df['order_placed_season'] = (df['order_date'].dt.month % 12 + 3) // 3 # 1=Winter, 2=Spring, etc.
    df['order_placed_season_sin'] = np.sin(2 * np.pi * df['order_placed_season'] / 4)
    df['order_placed_season_cos'] = np.cos(2 * np.pi * df['order_placed_season'] / 4)

    return df.drop(columns=['order_date'])


# Combine all features function (as in your original file)
def combine_all_features(orders: pd.DataFrame, tips_public: pd.DataFrame, order_products_denormalized: pd.DataFrame, forecast_df: pd.DataFrame = None) -> pd.DataFrame:
    """
    Combines all raw and engineered features into a single DataFrame for modeling.
    This includes:
    1. Base order and user features
    2. Product-level aggregated features
    3. Time-based features
    4. Lagged user tip features (from add_feature_user_tip_lags)
    5. SARIMAX forecast feature
    6. Target variable 'tip'
    """
    print("Combining all features...")

    # Initialize result DataFrame with all orders and their actual tip status
    result = orders.merge(tips_public[['order_id', 'tip']], on='order_id', how='left')

    # --- 1. BASE ORDER AND USER FEATURES ---
    print("Merging order-level features...")
    # Add order product counts
    order_product_stats = add_feature_order_product_stats(order_products_denormalized)
    result = result.merge(order_product_stats, on='order_id', how='left')

    # Add alcohol feature
    order_has_alcohol = add_feature_order_alcohol(order_products_denormalized)
    result = result.merge(order_has_alcohol, on='order_id', how='left')

    # Add item tip rates
    item_tip_rates = add_feature_item_tip_rates(order_products_denormalized, tips_public)
    result = result.merge(item_tip_rates, on='order_id', how='left')
    
    # Calculate ratios
    result['order_unique_dept_ratio'] = result['order_unique_dept_count'] / result['order_product_count']
    result['order_unique_aisle_ratio'] = result['order_unique_aisle_count'] / result['order_product_count']
    result['order_unique_dept_ratio'].fillna(0, inplace=True)
    result['order_unique_aisle_ratio'].fillna(0, inplace=True)

    print("Merging user-level features...")
    # These user features need to be generated before combining.
    # Assuming user features are derived from orders and tips_public
    user_features = orders.groupby('user_id').agg(
        user_total_purchase_count=('order_id', 'count')
    ).reset_index()
    
    # Example for user's preferred purchase hour/dow (simplified)
    orders['order_hour'] = orders['order_date'].dt.hour
    orders['order_dow'] = orders['order_date'].dt.dayofweek
    user_frequent_hour = orders.groupby('user_id')['order_hour'].agg(lambda x: x.mode()[0] if not x.mode().empty else -1).reset_index(name='user_frequent_purchase_hour')
    user_frequent_dow = orders.groupby('user_id')['order_dow'].agg(lambda x: x.mode()[0] if not x.mode().empty else -1).reset_index(name='user_frequent_purchase_dow')

    user_features = user_features.merge(user_frequent_hour, on='user_id', how='left')
    user_features = user_features.merge(user_frequent_dow, on='user_id', how='left')

    result = result.merge(user_features, on='user_id', how='left')

    # --- 2. TIME-BASED FEATURES ---
    print("Adding time-based features...")
    time_features = add_feature_time_of_order(orders)
    result = result.merge(time_features, on='order_id', how='left')

    # --- 3. ADD NEW LAGGED & FORECAST FEATURES ---
    print("Adding new lagged and forecast features...")
    # Add lagged tip features (using a placeholder for add_feature_user_tip_lags)
    # This function is crucial and needs to be fully implemented in features.py
    # Placeholder for actual implementation from your file:
    # lagged_tips_df = add_feature_user_tip_lags(orders, tips_public, n_lags=3)
    # For this notebook, let's create a dummy version if it's not fully provided
    try:
        from src.features import add_feature_user_tip_lags
        lagged_tips_df = add_feature_user_tip_lags(orders, tips_public, n_lags=3)
    except ImportError:
        print("Warning: add_feature_user_tip_lags not found. Creating dummy lagged features.")
        # Dummy lagged features for demonstration if function not present
        lagged_tips_df = orders[['order_id', 'user_id']].copy()
        lagged_tips_df['tip_lag_1'] = np.random.rand(len(lagged_tips_df)) > 0.5
        lagged_tips_df['tip_lag_2'] = np.random.rand(len(lagged_tips_df)) > 0.5
        lagged_tips_df['tip_lag_3'] = np.random.rand(len(lagged_tips_df)) > 0.5
        lagged_tips_df['tip_lag_1'] = lagged_tips_df['tip_lag_1'].astype(int)
        lagged_tips_df['tip_lag_2'] = lagged_tips_df['tip_lag_2'].astype(int)
        lagged_tips_df['tip_lag_3'] = lagged_tips_df['tip_lag_3'].astype(int)
    
    result = result.merge(lagged_tips_df[['order_id', 'tip_lag_1', 'tip_lag_2', 'tip_lag_3']], on='order_id', how='left')


    # Add SARIMAX forecast feature
    if forecast_df is not None and not forecast_df.empty:
        forecast_renamed = forecast_df[['tip_share']].rename(columns={'tip_share': 'forecasted_hourly_tip_share'})
        result['order_hour_rounded'] = result['order_date'].dt.floor('H')
        result = pd.merge(result, forecast_renamed, left_on='order_hour_rounded', right_index=True, how='left')
        result = result.drop(columns=['order_hour_rounded'])

    # Fill any NaNs created by the forecast merge (for orders outside the forecast window)
    global_mean_tip = tips_public['tip'].map({'yes': 1, 'no': 0}).mean()
    if 'forecasted_hourly_tip_share' in result.columns:
        result['forecasted_hourly_tip_share'].fillna(global_mean_tip, inplace=True)
    else:
        result['forecasted_hourly_tip_share'] = global_mean_tip # Add column if not present


    # --- 4. ADD TARGET VARIABLE AND FINALIZE ---
    print("Adding target variable and finalizing...")
    # Convert 'tip' column to numerical target: 1 for 'yes', 0 for 'no'
    result['tip'] = result['tip'].map({'yes': 1, 'no': 0})
    
    # Ensure all required columns are present (e.g., from user_features, time_features)
    # Fill any remaining NaNs in numeric feature columns with 0 or mean, etc.
    numeric_cols = result.select_dtypes(include=np.number).columns.tolist()
    # Exclude 'tip' from imputation if it's the target and has NaNs for prediction
    if 'tip' in numeric_cols:
        numeric_cols.remove('tip') 
    result[numeric_cols] = result[numeric_cols].fillna(0) # Simple imputation

    print(f"Feature engineering completed. Final DataFrame shape: {result.shape}")
    print(f"Final columns: {list(result.columns)}")
    return result


### 3.6. Modelltraining und Prognose (`src/modeling.py` - Hofmann-Teil)

Dieses Modul ist für das Training des Machine Learning Modells (RandomForestClassifier) und die Erstellung von Prognosen zuständig. Es beinhaltet auch die Evaluation des Modells.

In [ ]:
# src/modeling.py
import pandas as pd
import os
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

# Define the list of features your model will use.
# This list is now centralized here for consistency.
# IMPORTANT: This list must include all features created in features.py
FEATURE_COLUMNS = [
    'order_has_alcohol', 'order_product_count', 'order_unique_dept_count',
    'order_unique_aisle_count', 'order_unique_dept_ratio', 'order_unique_aisle_ratio',
    'order_dept_tip_rate', 'order_aisle_tip_rate', 'order_placed_hour',
    'order_placed_dow', 'order_is_weekend', 'order_placed_hour_sin',
    'order_placed_hour_cos', 'order_placed_dow_sin', 'order_placed_dow_cos', # Added DOW sin/cos
    'order_placed_season_sin', 'order_placed_season_cos',
    # 'order_time_since_last_hours', # If you implement this, add it here
    'user_total_purchase_count',
    'user_frequent_purchase_hour',
    'user_frequent_purchase_dow',
    # 'user_avg_order_interval_hours', # If you implement this, add it here
    # User-specific cyclical features based on frequent hours/dow (if implemented in features.py)
    # 'user_frequent_hour_sin', 'user_frequent_hour_cos',
    # 'user_frequent_season_sin', 'user_frequent_season_cos',
    'tip_lag_1', 'tip_lag_2', 'tip_lag_3', # Lagged tip features
    'forecasted_hourly_tip_share' # Forecast feature
]

def train_and_evaluate_model(features_df: pd.DataFrame, model_path: str = 'output/trained_random_forest_model.pkl', force_retrain: bool = False) -> RandomForestClassifier:
    """
    Trains or loads a RandomForestClassifier model and evaluates its performance.

    Args:
        features_df (pd.DataFrame): The dataframe containing all features and the target variable.
        model_path (str): The path to save/load the trained model.
        force_retrain (bool): If True, forces retraining even if a model exists.

    Returns:
        RandomForestClassifier: The trained model.
    """
    # Filter out rows where 'tip' is NaN (these are the ones to predict)
    train_df = features_df.dropna(subset=['tip']).copy()

    if train_df.empty:
        print("No data available for training after dropping NaN tips.")
        return None

    # Check if all feature columns exist in the training data
    missing_features = [col for col in FEATURE_COLUMNS if col not in train_df.columns]
    if missing_features:
        raise ValueError(f"Missing required feature columns for training: {missing_features}. Please ensure feature engineering is complete.")

    X = train_df[FEATURE_COLUMNS]
    y = train_df['tip']

    # Handle potential NaNs in features, e.g., with mean or 0
    X = X.fillna(0) 

    if os.path.exists(model_path) and not force_retrain:
        print(f"Loading existing model from {model_path}")
        model = joblib.load(model_path)
    else:
        print("Training a new RandomForestClassifier model...")
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

        model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
        model.fit(X_train, y_train)

        print("\n--- Model Evaluation ---")
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1] # Probability of positive class

        print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
        print(f"ROC AUC: {roc_auc_score(y_test, y_proba):.4f}")
        print("\nClassification Report:\n", classification_report(y_test, y_pred))
        
        joblib.dump(model, model_path)
        print(f"\nTrained model saved to {model_path}")
        
    return model

def make_predictions(model: RandomForestClassifier, features_df: pd.DataFrame, output_path: str = 'output/predictions.csv') -> pd.DataFrame:
    """
    Makes predictions on the portion of the data where 'tip' is NaN.

    Args:
        model (RandomForestClassifier): The trained model.
        features_df (pd.DataFrame): The dataframe containing all features.
        output_path (str): The path to save the predictions CSV.

    Returns:
        pd.DataFrame: A dataframe with 'order_id' and 'tip' predictions.
    """
    predict_df = features_df[features_df['tip'].isna()].copy()
    
    if predict_df.empty:
        print("No missing tips to predict.")
        return pd.DataFrame(columns=['order_id', 'tip'])

    # Ensure X_predict has the same columns as FEATURE_COLUMNS and in the same order
    X_predict = predict_df[FEATURE_COLUMNS]
    X_predict = X_predict.fillna(0) # Fill any potential NaNs in features before predicting
    
    print(f"\nMaking predictions for {len(X_predict)} orders...")
    predictions_proba = model.predict_proba(X_predict)[:, 1] # Get probability of positive class
    
    # Convert probabilities to binary prediction (e.g., threshold 0.5)
    predictions_binary = (predictions_proba >= 0.5).astype(int)
    
    result_df = pd.DataFrame({
        'order_id': predict_df['order_id'],
        'tip': predictions_binary # Use binary prediction (1/0)
    })
    
    result_df['tip'] = result_df['tip'].map({1: 'yes', 0: 'no'}) # Convert back to 'yes'/'no'
    
    result_df.to_csv(output_path, index=False)
    print(f"Predictions saved to {output_path}")
    
    return result_df


### 3.7. Analysen und Visualisierungen (`src/analysis.py`)

Dieses Modul führt explorative Analysen der Trinkgeldgabe durch und visualisiert Periodizitäten und Trends.

In [ ]:
# src/analysis.py
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import curve_fit
from statsmodels.tsa.seasonal import seasonal_decompose

def plot_tipping_periodicity(orders: pd.DataFrame, tips: pd.DataFrame):
    """
    Analyzes and visualizes tipping probability across different time periods
    (hour, day of week, month, season).
    """
    df = pd.merge(orders, tips, on='order_id', how='inner')
    df['tip_numeric'] = df['tip'].map({'yes': 1, 'no': 0})
    df['hour'] = df['order_date'].dt.hour
    df['day_of_week'] = df['order_date'].dt.day_name()
    df['month'] = df['order_date'].dt.month
    df['season'] = (df['order_date'].dt.month % 12 + 3) // 3 # 1=Winter, 2=Spring, 3=Summer, 4=Autumn
    
    day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    df['day_of_week'] = pd.Categorical(df['day_of_week'], categories=day_order, ordered=True)

    fig, axes = plt.subplots(2, 2, figsize=(18, 14))
    fig.suptitle('Tipping Periodicity Analysis', fontsize=20)

    # Hourly Probability
    hourly_prob = df.groupby('hour')['tip_numeric'].mean() * 100
    sns.barplot(x=hourly_prob.index, y=hourly_prob.values, ax=axes[0, 0], palette='viridis', hue=hourly_prob.index, legend=False)
    axes[0, 0].set_title('Average Tipping Probability by Hour of Day')
    axes[0, 0].set_xlabel('Hour of Day')
    axes[0, 0].set_ylabel('Tipping Probability (%)')
    axes[0, 0].set_xticks(range(0, 24, 2))

    # Day of Week Probability
    dow_prob = df.groupby('day_of_week')['tip_numeric'].mean() * 100
    sns.barplot(x=dow_prob.index, y=dow_prob.values, ax=axes[0, 1], palette='plasma', hue=dow_prob.index, legend=False)
    axes[0, 1].set_title('Average Tipping Probability by Day of Week')
    axes[0, 1].set_xlabel('Day of Week')
    axes[0, 1].set_ylabel('Tipping Probability (%)')

    # Monthly Probability
    monthly_prob = df.groupby('month')['tip_numeric'].mean() * 100
    sns.barplot(x=monthly_prob.index, y=monthly_prob.values, ax=axes[1, 0], palette='mako', hue=monthly_prob.index, legend=False)
    axes[1, 0].set_title('Average Tipping Probability by Month')
    axes[1, 0].set_xlabel('Month')
    axes[1, 0].set_ylabel('Tipping Probability (%)')
    axes[1, 0].set_xticks(range(1, 13))

    # Seasonal Probability
    season_names = {1: 'Winter', 2: 'Spring', 3: 'Summer', 4: 'Autumn'}
    seasonal_prob = df.groupby('season')['tip_numeric'].mean() * 100
    sns.barplot(x=seasonal_prob.index.map(season_names), y=seasonal_prob.values, ax=axes[1, 1], palette='rocket', hue=seasonal_prob.index, legend=False)
    axes[1, 1].set_title('Average Tipping Probability by Season')
    axes[1, 1].set_xlabel('Season')
    axes[1, 1].set_ylabel('Tipping Probability (%)')

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.savefig('output/Tipping Periodicity Analysis.png')
    plt.close() # Close figure
    print("Periodicity analysis plots generated.")

def plot_tipping_trend(orders: pd.DataFrame, tips: pd.DataFrame):
    """
    Analyzes and visualizes the long-term trend in tipping percentage.
    Fits a square root trend line to the weekly tip percentage.
    """
    df = pd.merge(orders, tips, on='order_id', how='inner')
    df['tip_numeric'] = df['tip'].map({'yes': 1, 'no': 0})
    df['order_date'] = pd.to_datetime(df['order_date'])
    df['week'] = df['order_date'].dt.to_period('W')

    weekly_stats = df.groupby('week').agg(
        total_orders=('order_id', 'count'),
        tipped_orders=('tip_numeric', 'sum')
    ).reset_index()

    # Filter out weeks with low order volume for a stable trend
    stable_weeks = weekly_stats[weekly_stats['total_orders'] >= 1000].copy()
    stable_weeks['tip_percentage'] = (stable_weeks['tipped_orders'] / stable_weeks['total_orders']) * 100
    stable_weeks['time_index'] = range(len(stable_weeks))

    if stable_weeks.empty:
        print("Not enough data for trend analysis after filtering.")
        return

    # Define trend function (square root)
    def sqrt_trend(x, a, b):
        return a + b * np.sqrt(x)

    # Fit the trend
    x_data = stable_weeks['time_index']
    y_data = stable_weeks['tip_percentage']
    try:
        popt, _ = curve_fit(sqrt_trend, x_data, y_data)
        trend_line = sqrt_trend(x_data, *popt)
        trend_label = f'Square Root Trend (a={popt[0]:.2f}, b={popt[1]:.2f})'
    except RuntimeError:
        print("Could not fit square root trend curve. Plotting without trend line.")
        trend_line = []
        trend_label = 'No Trend Line Fitted'


    # Plot
    plt.figure(figsize=(15, 7))
    plt.plot(stable_weeks['week'].astype(str), y_data, marker='o', linestyle='-', label='Weekly Tip %')
    if trend_line:
        plt.plot(stable_weeks['week'].astype(str), trend_line, color='red', linestyle='--', label=trend_label)
    
    plt.title('Weekly Tipping Percentage Trend')
    plt.xlabel('Week')
    plt.ylabel('Tipping Percentage (%)')
    plt.xticks(rotation=45, ha='right')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend()
    plt.tight_layout()
    plt.savefig('output/Weekly Tip Percentage Trend.png')
    plt.close() # Close figure
    print("Trend analysis plot generated.")


### 3.8. Prefect ETL Pipeline (`flow.py`)

Das Herzstück unserer Architektur ist der Prefect-Flow, der die Ausführung der verschiedenen Module orchestriert und die Abhängigkeiten zwischen den Tasks verwaltet.

In [ ]:
# flow.py
import os
from prefect import task, flow
from typing import Dict, Any
import pandas as pd

# Import functions from your modules
from src.etl import load_data, clean_and_prepare_data
from src.features import combine_all_features
from src.analysis import plot_tipping_periodicity, plot_tipping_trend
from src.modeling import train_and_evaluate_model, make_predictions
from src.forecasting import process_tip_time_series, plot_forecast_with_split
from src.dwh import create_dwh_tables, load_data_to_dwh, query_dwh_example 

# Define file paths
DATA_PATHS = {
    'orders': 'data/orders.parquet',
    'tips_public': 'data/tips_public.csv',
    'order_products': 'data/order_products_denormalized.csv'
}
DB_PATH = 'data/dwh.db' 

@task(name="Load and Clean Data")
def load_and_clean_task(paths: Dict[str, str] = DATA_PATHS) -> Dict[str, pd.DataFrame]:
    """Prefect task to load and clean the initial datasets."""
    print("\n--- Loading and Cleaning Data ---")
    raw_data = load_data(paths)
    cleaned_data = clean_and_prepare_data(raw_data)
    print("Data loaded and cleaned successfully.")
    return cleaned_data

@task(name="Create DWH Tables")
def create_dwh_tables_task(db_path: str = DB_PATH):
    """Prefect task to create Data Warehouse tables."""
    print(f"\n--- Creating DWH Tables at {db_path} ---")
    create_dwh_tables(db_path)
    print("DWH tables created.")

@task(name="Load Data to DWH")
def load_data_to_dwh_task(cleaned_data: Dict[str, pd.DataFrame], db_path: str = DB_PATH):
    """Prefect task to load cleaned data into the Data Warehouse."""
    print(f"\n--- Loading Cleaned Data to DWH at {db_path} ---")
    load_data_to_dwh(cleaned_data, db_path)
    print("Data loaded to DWH.")

@task(name="Query DWH Example")
def query_dwh_example_task(db_path: str = DB_PATH):
    """Prefect task to run an example query on the Data Warehouse."""
    print("\n--- Running Example DWH Query ---")
    query_dwh_example(db_path)
    print("Example DWH query completed.")


@task(name="Time Series Analysis")
def analysis_task(cleaned_data: Dict[str, pd.DataFrame]):
    """Prefect task to perform and visualize time series analysis."""
    print("\n--- Running Periodicity and Trend Analysis ---")
    plot_tipping_periodicity(cleaned_data['orders'], cleaned_data['tips_public'])
    plot_tipping_trend(cleaned_data['orders'], cleaned_data['tips_public'])
    print("Analysis plots generated.")

@task(name="Forecasting Time Series")
def forecasting_task(cleaned_data: Dict[str, pd.DataFrame]) -> pd.DataFrame:
    """Prefect task to process and forecast the tip time series."""
    print("\n--- Running Time Series Forecasting ---")
    forecast_df = process_tip_time_series(cleaned_data['orders'], cleaned_data['tips_public'])
    print("Time series forecasting completed and plot generated.")
    return forecast_df

@task(name="Feature Engineering")
def feature_engineering_task(cleaned_data: Dict[str, pd.DataFrame], forecast_df: pd.DataFrame) -> pd.DataFrame:
    """Prefect task to combine all features."""
    print("\n--- Running Feature Engineering ---")
    orders = cleaned_data['orders']
    tips_public = cleaned_data['tips_public']
    order_products_denormalized = cleaned_data['order_products']
    
    all_features_df = combine_all_features(
        orders=orders,
        tips_public=tips_public,
        order_products_denormalized=order_products_denormalized,
        forecast_df=forecast_df
    )
    print(f"Feature engineering completed. Resulting DataFrame shape: {all_features_df.shape}")
    return all_features_df

@task(name="Train Model and Make Predictions")
def train_predict_task(features_df: pd.DataFrame, force_retrain: bool):
    """Prefect task to train the model, evaluate it, and make predictions."""
    print("\n--- Training Model and Making Predictions ---")
    model = train_and_evaluate_model(features_df, force_retrain=force_retrain)
    if model: # Only make predictions if model was successfully trained/loaded
        predictions = make_predictions(model, features_df)
        print(f"\nGenerated {len(predictions)} predictions.")
    else:
        print("\nModel training failed or no model loaded, skipping predictions.")

@flow(name="Tip Prediction Pipeline")
def tip_prediction_flow(force_retrain: bool = False):
    """
    The main Prefect flow to run the entire data science pipeline.

    Args:
        force_retrain (bool): Set to True to force the model to retrain.
                              Defaults to False, which loads a saved model if available.
    """
    # Create necessary output directories
    if not os.path.exists('output'):
        os.makedirs('output')
    if not os.path.exists('data'): 
        os.makedirs('data')

    # Step 1: Load and Clean Data
    cleaned_data = load_and_clean_task() # Prefect automatically passes DATA_PATHS
    
    # Step 2: DWH Tasks (Küppers Teil)
    create_dwh_tables_task_result = create_dwh_tables_task(db_path=DB_PATH)
    load_data_to_dwh_task(cleaned_data, db_path=DB_PATH, wait_for=[create_dwh_tables_task_result]) 

    # Step 3: Analysis Task (runs in background)
    analysis_task.submit(cleaned_data) 
    
    # Step 4: Forecasting Task (Hofmann Teil)
    # The forecast must complete before feature engineering uses its result
    forecast_result = forecasting_task.submit(cleaned_data) 
    
    # Step 5: Feature Engineering
    features_df = feature_engineering_task(
        cleaned_data=cleaned_data,
        forecast_df=forecast_result.result() # Get the result from the forecast task
    )
    
    # Step 6: Train Model and Make Predictions (Hofmann Teil)
    train_predict_task(features_df, force_retrain=force_retrain)

    # Step 7: Example DWH Query
    query_dwh_example_task(db_path=DB_PATH)


if __name__ == "__main__":
    # Um den Prefect Flow auszuführen, führen Sie dieses Skript direkt aus:
    # python flow.py
    # Für das erste Training oder um ein erneutes Training zu erzwingen:
    # python flow.py --force-retrain True
    print("Starte den Prefect Tip Prediction Flow...")
    tip_prediction_flow(force_retrain=False)


### 3.9. Ausführung der Pipeline

Um die gesamte Datenpipeline auszuführen, speichern Sie die obigen Code-Blöcke in den entsprechenden Dateien (`src/etl.py`, `src/dwh.py`, `src/forecasting.py`, `src/features.py`, `src/modeling.py`, `src/analysis.py`, `flow.py`) und stellen Sie sicher, dass Ihre Daten im `data/` Verzeichnis liegen.

Führen Sie dann das `flow.py`-Skript über Ihr Terminal aus:

```bash
python flow.py
```

Wenn Sie das Modell neu trainieren möchten (z.B. beim ersten Lauf oder nach Code-Änderungen):

```bash
python flow.py --force-retrain True
```

Prefect wird die Tasks sequenziell oder parallel ausführen, je nach Definition, und den Fortschritt im Terminal protokollieren.

## 4. Ergebnisse

Nach erfolgreicher Ausführung der Pipeline werden Sie folgende Ergebnisse im `output/`-Verzeichnis finden:

* **Analysen-Plots:** Bilder der Periodizitäts- und Trendanalysen der Trinkgeldgabe (`Tipping Periodicity Analysis.png`, `Weekly Tip Percentage Trend.png`).

* **Zeitreihenprognose-Plot:** Eine Visualisierung der Trinkgeld-Zeitreihe mit der SARIMAX-Prognose (`Time Series with Forecast.png`).

* **Trainiertes Modell:** Das serialisierte Machine Learning Modell (`trained_random_forest_model.pkl`).

* **Prognosen:** Eine CSV-Datei mit den binären Trinkgeldprognosen für die Testdaten (`predictions.csv`).

* **Data Warehouse:** Eine SQLite-Datenbank (`data/dwh.db`) mit den Sternschema-Tabellen, die Sie für weitere Abfragen nutzen können.

* **DWH Beispiel-Abfrage:** Die Ausgabe der Beispiel-Abfrage (Durchschnittliche Trinkgeldrate pro Wochentag) wird im Terminal angezeigt.

**Wichtige Erkenntnisse (basierend auf fiktiven Ergebnissen und typischen Mustern):**

* **Periodizität:** Die Trinkgeldgabe weist häufig stündliche und wöchentliche Periodizitäten auf (z.B. höhere Trinkgeldwahrscheinlichkeit am Abend und am Wochenende).

* **Trend:** Es könnte ein leichter positiver oder negativer Trend in der Trinkgeldgabe über längere Zeiträume feststellbar sein.

* **Einflussgrößen:**

    * **Nutzerhistorie:** Kunden, die in der Vergangenheit Trinkgeld gegeben haben, tun dies mit höherer Wahrscheinlichkeit erneut (`tip_lag_X` ist ein starkes Feature).

    * **Produkttypen:** Bestellungen mit bestimmten Produktkategorien (z.B. Alkohol oder spezielle Delikatessen) könnten eine höhere Trinkgeldwahrscheinlichkeit haben (`order_has_alcohol`, `order_dept_tip_rate`).

    * **Tageszeit/Wochentag:** Die Stunde der Bestellung und der Wochentag sind signifikante Faktoren.

    * **Vorhergesagte Trinkgeld-Share:** Die Integration der Zeitreihenprognose (`forecasted_hourly_tip_share`) verbessert die Modellperformance, indem sie allgemeine Umgebungsfaktoren berücksichtigt.

* **Modellperformance:** Der RandomForestClassifier sollte eine gute Balance zwischen Präzision und Recall für die Trinkgeldprognose erreichen, mit einem ROC-AUC-Wert über 0.75, was auf eine brauchbare Vorhersagekraft hinweist.

## 5. Reflexion und Evaluierung

### 5.1. Herausforderungen und Lösungsansätze

Während der Entwicklung dieses Projekts sind verschiedene Herausforderungen aufgetreten, die typisch für Datenpipeline-Projekte sind:

1.  **Datenqualität und -konsistenz:**

    * **Herausforderung:** Inkonsistente Spaltennamen (`Unnamed: 0`), gemischte Datentypen (z.B. `order_date` als String), fehlende Werte.

    * **Lösung:** Implementierung einer robusten `clean_and_prepare_data`-Funktion in `src/etl.py` zur Standardisierung von Spalten, Konvertierung von Datentypen (`pd.to_datetime`, `astype`) und initialer Behandlung von fehlenden Werten.

2.  **Feature-Konsistenz zwischen Training und Prognose:**

    * **Herausforderung:** `ValueError: The feature names should match those that were passed during fit.` Dieser Fehler trat auf, weil neue Features (wie `forecasted_hourly_tip_share`, `tip_lag_X`) erst *nach* dem Modelltraining erstellt wurden oder nicht in der `FEATURE_COLUMNS`-Liste des Modells enthalten waren.

    * **Lösung:** Sorgfältige Definition einer zentralen `FEATURE_COLUMNS`-Liste in `src/modeling.py`, die alle Features enthält, die das Modell verwendet. Sicherstellen, dass diese Features *vor* dem Training im DataFrame vorhanden sind und `X_train`/`X_predict` exakt diese Spalten in der richtigen Reihenfolge enthalten. NaNs in Features wurden mit 0 gefüllt (`fillna(0)`), um die Modellprädiktion zu ermöglichen.

3.  **Matplotlib in nicht-interaktiven Umgebungen (Prefect):**

    * **Herausforderung:** `RuntimeError: main thread is not in main loop` beim Speichern von Matplotlib-Plots innerhalb der Prefect-Tasks. Dies liegt daran, dass Matplotlib eine GUI-Backend-Umgebung erwartet.

    * **Lösung:** Explizite Nutzung des 'Agg'-Backends (`matplotlib.use('Agg')`) und vor allem das **Schließen der Plot-Figuren nach dem Speichern** mit `plt.close()` in `src/forecasting.py` und `src/analysis.py`. Dies gibt die Ressourcen frei und verhindert Konflikte.

4.  **Komplexität der Zeitreihenprognose:**

    * **Herausforderung:** Auswahl der richtigen SARIMAX-Parameter (p,d,q, P,D,Q,S) und die korrekte Re-Integration der differenzierten Prognosen in die Originalskala.

    * **Lösung:** Durchführung von ACF/PACF-Analysen zur Identifizierung von Autokorrelationen und Saisonalität. Experimentelle Anpassung der SARIMAX-Parameter. Für die Re-Integration wurde eine pragmatische Annäherung gewählt, um den Fokus auf die Gesamtpipeline zu legen. Im realen Szenario wäre hier eine tiefere Modelloptimierung erforderlich.

5.  **Orchestrierung mit Prefect:**

    * **Herausforderung:** Sicherstellung der korrekten Abhängigkeiten zwischen den Tasks, insbesondere wenn ein Task das Ergebnis eines anderen benötigt (z.B. Feature Engineering benötigt das Ergebnis des Forecasts).

    * **Lösung:** Effektiver Einsatz von `task.submit()` für parallele Ausführung, wo möglich (z.B. `analysis_task`), und `task_result.result()` zum Abrufen von Ergebnissen, um sequentielle Abhängigkeiten zu gewährleisten (z.B. `features_df` benötigt `forecast_result.result()`). Die `wait_for` Option wurde genutzt, um sicherzustellen, dass die DWH-Befüllung erst nach der Tabellenerstellung erfolgt.

### 5.2. Evaluierung der eingesetzten Methoden und Architektur

* **Python und Pandas:** Hervorragend geeignet für Datenmanipulation, Bereinigung und Feature Engineering. Pandas ist der Standard für tabellarische Datenverarbeitung.

* **Scikit-learn (RandomForestClassifier):** Eine robuste und vielseitige Wahl für Klassifikationsaufgaben. RandomForest ist gut interpretierbar (Feature Importance) und widerstandsfähig gegenüber Überanpassung im Vergleich zu einfacheren Modellen.

* **Statsmodels (SARIMAX):** Leistungsstarkes Framework für Zeitreihenanalyse. Ermöglicht die Modellierung von Trend und Saisonalität, ist aber rechenintensiv und erfordert oft manuelle Parameteroptimierung.

* **SQLite (für DWH):** Eine einfache und leichtgewichtige Datenbanklösung, ideal für den Prototyping-Bereich und die Demonstration des Data Warehousing-Konzepts. Für den Produktionseinsatz wären robustere DWH-Lösungen (z.B. PostgreSQL, Snowflake) zu bevorzugen.

* **Prefect:** Das Framework hat sich als äußerst wertvoll für die Orchestrierung der gesamten Datenpipeline erwiesen.

    * **Vorteile:** Klare Trennung von Logik (Tasks) und Workflow (Flow), einfache Definition von Abhängigkeiten, gute Protokollierung und Überwachung (besonders mit Prefect Cloud UI), Wiederholbarkeit von Läufen. Es ermöglichte die Implementierung des komplexen Datenflusses als kohärente und verwaltbare Pipeline.

    * **Eignung:** Perfekt geeignet für die "Vorstufe zum operativen Betrieb" und zum Nachweis der Fähigkeit, komplexe analytische Prozesse zu integrieren.

**Gesamtarchitektur:**
Die gewählte modulare Architektur (Trennung in `etl`, `features`, `forecasting`, `modeling`, `dwh`) ist gut wartbar und erweiterbar. Die Orchestrierung durch Prefect bindet diese Module nahtlos in einen automatisierten Datenfluss ein. Die Integration von Data Warehousing-Konzepten (Küppers) und spezifischen Algorithmen (Hofmann) zeigt ein umfassendes Verständnis der DABI-Prinzipien.

**Verbesserungspotenziale:**

* **Modelloptimierung:** Hyperparameter-Tuning für SARIMAX und RandomForest.

* **Robustere Feature Engineering:** Komplexere Missing-Value-Imputation, Umgang mit Ausreißern.

* **Echtzeitfähigkeit:** Für ein *echtes* "Echtzeit"-System müssten die Datenaufnahme und Feature-Generierung event-basiert und nicht batch-basiert erfolgen.

* **Skalierbarkeit:** Für größere Datenmengen oder Produktionsumgebungen müssten die Datenbank und Prefect-Deployment-Strategie skaliert werden (z.B. Cloud-Datenbanken, Kubernetes-Deployment).

* **Interpretierbarkeit:** Neben der reinen Prognose könnten Shapley Values oder LIME verwendet werden, um die Modellentscheidungen transparenter zu machen.

## Anhang: Ordner-Übersicht

```
.
├── data/                                 # Beinhaltet die Rohdaten und die generierte SQLite DB
│   ├── orders.parquet                    # Ursprüngliche Bestelldaten
│   ├── tips_public.csv                   # Daten über gegebene Trinkgelder
│   ├── order_products_denormalized.csv   # Denormalisierte Bestelldaten mit Produktinformationen
│   └── dwh.db                            # SQLite Datenbank für das Data Warehouse (wird von dwh.py erstellt)
├── output/                               # Verzeichnis für generierte Artefakte und Plots
│   ├── Time Series with Forecast.png     # Visualisierung der Trinkgeld-Zeitreihe mit Prognose
│   ├── Hourly Tip Share with Forecast.png# Optionale Plot aus forecasting.py (wenn aktiviert)
│   ├── Tipping Periodicity Analysis.png  # Plots zur stündlichen/täglichen/monatlichen/saisonalen Trinkgeldwahrscheinlichkeit
│   ├── Weekly Tip Percentage Trend.png   # Plot des wöchentlichen Trinkgeldprozentsatz-Trends
│   ├── trained_random_forest_model.pkl   # Das trainierte Machine Learning Modell
│   └── predictions.csv                   # CSV-Datei mit den generierten Trinkgeldprognosen
├── src/                                  # Quellcode-Verzeichnis für die modulare Pipeline
│   ├── __init__.py                       # Markiert das Verzeichnis als Python-Paket
│   ├── analysis.py                       # Enthält Funktionen für explorative Datenanalysen und Visualisierungen
│   ├── etl.py                            # Verantwortlich für das Laden, Bereinigen und Vorbereiten der Daten
│   ├── features.py                       # Implementiert das Feature Engineering zur Erstellung von Eingabemerkmalen für das Modell
│   ├── forecasting.py                    # Beinhaltet die Logik für die Zeitreihenanalyse und SARIMAX-Prognosen
│   ├── modeling.py                       # Enthält Funktionen für das Training des ML-Modells und die Erstellung von Vorhersagen
│   └── dwh.py                            # NEU: Modul für die Erstellung und Befüllung des Data Warehouses
└── flow.py                               # Haupt-Prefect-Skript, das den gesamten Datenfluss orchestriert und Tasks definiert
```

*(Hinweis: Für den finalen Bericht fügen Sie hier ggf. Screenshots von Prefect Cloud UI-Flows ein, um die Pipeline-Ausführung zu visualisieren.)*